## Importing Libraries and Definitions

In [ ]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
Holistic = mp.solutions.holistic.Holistic
def extract_landmarks(landmarks):
    return np.array([[l.x, l.y] for l in landmarks],dtype=np.float32)

### Saving Images by Clicking 

In [ ]:
import cv2

cam = cv2.VideoCapture(0)

if not cam.isOpened():
    print("Error: Camera not opened")
    exit()

cv2.namedWindow("Camera")

counter = 0
current_frame = None

def mouse_click(event, x, y, flags, param):
    global counter, current_frame
    if event == cv2.EVENT_LBUTTONDOWN:
        counter += 1
        filename = f"data/scissors/scissors{counter}.png"
        cv2.imwrite(filename, current_frame)
        print(f"Saved {filename}")

cv2.setMouseCallback("Camera", mouse_click)

while True:
    ret, frame = cam.read()
    if not ret:
        print("Failed to grab frame")
        break

    current_frame = frame.copy()
    # current_frame = cv2.flip(current_frame, 1)
    # rgb_frame = cv2.cvtColor(current_frame, cv2.COLOR_BGR2RGB)
    # results = holistic.process(rgb_frame)
        # Make predictions
        
    cv2.imshow("Camera", frame)
    # drawing_utils.draw_landmarks(
    #         frame,
    #         results.right_hand_landmarks,
    #         mp.solutions.holistic.HAND_CONNECTIONS,
    #         connection_drawing_spec=drawing_styles.get_default_hand_connections_style()
    #     )
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()


ValueError: _graph is None in SolutionBase

: 

### Pose Processing of Image (irrelevant)

In [8]:
import os
import pickle
dataset = []
target = []
folder_path_paper = "data/paper/" 
folder_path = folder_path_paper
all_files = os.listdir(folder_path)  # lists files and folders
files_only = [f for f in all_files if os.path.isfile(os.path.join(folder_path, f))]
counter = 0
L = len(files_only)
model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

for f in files_only:
    counter += 1
    print(100*counter/L,end="\r")
    image = plt.imread(folder_path+f)
    img_model = (image[..., :3] * 255).astype(np.uint8)
    model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
    results = model.process(img_model)
    if results.pose_landmarks is not None:
        ldk = extract_landmarks(results.pose_landmarks.landmark)
        ldk_temp = extract_landmarks(results.pose_landmarks.landmark).flatten()
        dataset.append(ldk_temp)

with open(folder_path+'dataset_paper.pkl', 'wb') as file:  # 'wb' = write binary
    pickle.dump(dataset, file)

### Right-hand Processing of Image (irrelevant)

In [11]:
import os
import pickle
dataset = []
target = []
gestures = ["rock","paper","scissors"]
folder_path_rock = "data/rock/" 
folder_path_scissors = "data/scissors/" 
folder_paths = [folder_path_rock,folder_path_scissors]
# folder_path = folder_path_paper
for gesture in gestures:
    folder_path = "data/"+gesture+"/"
    all_files = os.listdir(folder_path)  # lists files and folders
    files_only = [f for f in all_files 
              if os.path.isfile(os.path.join(folder_path, f)) and f.lower().endswith('.png')]

    counter = 0
    L = len(files_only)
    model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

    for f in files_only:
        counter += 1
        print(100*counter/L,end="\r")
        image = plt.imread(folder_path+f)
        img_model = (image[..., :3] * 255).astype(np.uint8)
        model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
        results = model.process(img_model)
        if results.right_hand_landmarks is not None:
            ldk_temp = extract_landmarks(results.right_hand_landmarks.landmark).flatten()
            dataset.append(ldk_temp)

    with open(folder_path+'dataset_RightHand_'+gesture+'.pkl', 'wb') as file:  # 'wb' = write binary
        pickle.dump(dataset, file)

: 

### Loading Individual Datasets

In [ ]:

import pickle
import numpy as np 
# with open("data/paper/dataset_RightHand_paper.pkl", "rb") as f:
#     paper = pickle.load(f)
# with open("data/rock/dataset_RightHand_rock.pkl", "rb") as f:
#     rock = pickle.load(f)
# with open("data/scissors/dataset_RightHand_scissors.pkl", "rb") as f:
#     scissors = pickle.load(f)

with open("data/paper/Manual_dataset_RightHand_paper.pkl", "rb") as f:
    paper = pickle.load(f)
with open("data/rock/Manual_dataset_RightHand_rock.pkl", "rb") as f:
    rock = pickle.load(f)
with open("data/scissors/Manual_dataset_RightHand_scissors.pkl", "rb") as f:
    scissors = pickle.load(f)

dataset = np.concatenate((np.array(rock),np.array(paper),np.array(scissors)),axis=0)
target = ['rock']*len(rock)+['paper']*len(paper)+['scissors']*len(scissors)
columns = []
for col in range(int(dataset.shape[1]/2)):
    columns.append(f"x{col}")
    columns.append(f"y{col}")

### Converting to DataFrame and Saving to Pickle

In [ ]:
import pandas as pd
data = pd.DataFrame(data= dataset,columns=columns,index = range(len(dataset)))
data["target"] = target 
data.to_pickle("Manual_dataset_total_RightHand")
data

## Neural Network Model (93% accuracy)

### Building Model

In [73]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input,Dropout

landmarks_of_interest = range(int((data.shape[1]-1)/2))
length_inputs = len(landmarks_of_interest)
# Create the model
model = Sequential([
    Input((2*length_inputs,)),
    Dense(128,activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')  # Output layer for 3 classes
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Use for one-hot encoded labels
    metrics=['accuracy']
)

# Model summary
model.summary()


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_22 (Dense)                │ (None, 128)            │         5,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,939 (62.26 KB)

 Trainable params: 15,939 (62.26 KB)

 Non-trainable params: 0 (0.00 B)

### Loading & Splitting into Train/Test

In [68]:
df = pd.read_pickle("Manual_dataset_total_RightHand")
for index, row in df.iterrows():
    temp = row.copy()               # important!
    temp[0:-1:2] -=  temp.iloc[0:-1:2].min()
    temp[1:-1:2] -=  temp.iloc[1:-1:2].min()
    df.loc[index] = temp
from sklearn.model_selection import train_test_split
X = df[df.columns[:-1]].values
y = df[df.columns[-1]].values
dummies = pd.get_dummies(y).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, dummies, test_size=0.15, random_state=42)


C:\Users\mehdi\AppData\Local\Temp\ipykernel_5684\3382274615.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.2660316377878189' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[index] = temp
C:\Users\mehdi\AppData\Local\Temp\ipykernel_5684\3382274615.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.15149056166410446' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[index] = temp
C:\Users\mehdi\AppData\Local\Temp\ipykernel_5684\3382274615.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.16729766875505447' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  df.loc[index] = temp
C:\Users\mehdi\Ap

### Training NN Model

In [74]:
from tensorflow.keras.callbacks import EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',      # Metric to monitor
    patience=3,              # Stop after 5 epochs of no improvement
    verbose=1,               # Print messages when stopping
    restore_best_weights=True  # Restore model weights from the epoch with the best value of monitored metric
)
history = model.fit(
    X_train,
    y_train.values,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test),
    callbacks = [early_stop]
)

Epoch 1/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.3814 - loss: 1.0948 - val_accuracy: 0.5217 - val_loss: 1.0722
Epoch 2/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4691 - loss: 1.0752 - val_accuracy: 0.5362 - val_loss: 1.0442
Epoch 3/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5438 - loss: 1.0441 - val_accuracy: 0.5217 - val_loss: 1.0117
Epoch 4/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5155 - loss: 1.0174 - val_accuracy: 0.8116 - val_loss: 0.9554
Epoch 5/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6057 - loss: 0.9809 - val_accuracy: 0.7246 - val_loss: 0.9099
Epoch 6/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6443 - loss: 0.9342 - val_accuracy: 0.7971 - val_loss: 0.8440
Epoch 7/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6778 - loss: 0.8785 - val_accuracy: 0.7681 - val_loss: 0.7661
Epoch 8/30
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7010 - loss: 0.8230 - val_accuracy: 0.8261 - val_loss

### Saving Model to .pkl

In [75]:
with open('model.pkl', 'wb') as file:  # 'wb' = write binary
        pickle.dump(model, file)

## RandomForest Model (66% Accuracy)

In [13]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
params = {'n_estimators': range(100,800,100),
          'max_depth' : range(2,5,2)}
cv = RandomizedSearchCV(rf,params,verbose=2,scoring="accuracy",n_jobs=-1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42)
history = cv.fit(X_train,y_train)


Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [74]:
pd.DataFrame(history.cv_results_)


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,1.195370,0.023129,0.088505,0.006555,600,4,"{'n_estimators': 600, 'max_depth': 4}",0.706897,0.603448,0.620690,0.684211,0.614035,0.645856,0.041572,8
1,0.192483,0.021954,0.012430,0.001859,100,2,"{'n_estimators': 100, 'max_depth': 2}",0.620690,0.637931,0.689655,0.596491,0.719298,0.652813,0.045173,3
2,1.524023,0.075228,0.093853,0.009314,700,2,"{'n_estimators': 700, 'max_depth': 2}",0.620690,0.672414,0.689655,0.561404,0.701754,0.649183,0.051882,5
3,0.840734,0.050709,0.052605,0.013381,400,2,"{'n_estimators': 400, 'max_depth': 2}",0.655172,0.655172,0.706897,0.561404,0.719298,0.659589,0.055646,2
4,0.231173,0.020247,0.013568,0.004035,100,4,"{'n_estimators': 100, 'max_depth': 4}",0.724138,0.603448,0.603448,0.684211,0.596491,0.642347,0.052099,9
5,0.932329,0.055516,0.056698,0.005751,400,4,"{'n_estimators': 400, 'max_depth': 4}",0.706897,0.637931,0.637931,0.684211,0.596491,0.652692,0.038797,4
6,1.126487,0.031704,0.067970,0.007856,500,4,"{'n_estimators': 500, 'max_depth': 4}",0.706897,0.603448,0.586207,0.684211,0.614035,0.638959,0.047599,10
7,0.980709,0.092516,0.066166,0.005243,500,2,"{'n_estimators': 500, 'max_depth': 2}",0.603448,0.689655,0.706897,0.578947,0.666667,0.649123,0.049597,7
8,0.417215,0.004333,0.028461,0.001564,200,2,"{'n_estimators': 200, 'max_depth': 2}",0.603448,0.672414,0.706897,0.561404,0.701754,0.649183,0.057326,5
9,0.986080,0.029130,0.052241,0.013281,600,2,"{'n_estimators': 600, 'max_depth': 2}",0.620690,0.689655,0.706897,0.596491,0.701754,0.663097,0.045503,1


## Training XgBoost Model (66% Accuracy)

In [77]:
ybis = y
ybis[ybis=='rock'] = 0 
ybis[ybis=='paper'] = 1 
ybis[ybis=='scissors'] = 2 
X_train, X_test, y_train, y_test = train_test_split(
    X, ybis, test_size=0.15, random_state=42)

In [78]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


model = xgb.XGBClassifier(
    objective='multi:softprob',  # probabilities for each class
    num_class=3,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=True
)


[0]	validation_0-mlogloss:1.04243
[1]	validation_0-mlogloss:1.01166
[2]	validation_0-mlogloss:0.98542
[3]	validation_0-mlogloss:0.95266
[4]	validation_0-mlogloss:0.92429
[5]	validation_0-mlogloss:0.90269
[6]	validation_0-mlogloss:0.88242
[7]	validation_0-mlogloss:0.85825
[8]	validation_0-mlogloss:0.84342
[9]	validation_0-mlogloss:0.82595
[10]	validation_0-mlogloss:0.80786
[11]	validation_0-mlogloss:0.79072
[12]	validation_0-mlogloss:0.77851
[13]	validation_0-mlogloss:0.76497
[14]	validation_0-mlogloss:0.75371
[15]	validation_0-mlogloss:0.74318
[16]	validation_0-mlogloss:0.73348
[17]	validation_0-mlogloss:0.72282
[18]	validation_0-mlogloss:0.71443
[19]	validation_0-mlogloss:0.70671
[20]	validation_0-mlogloss:0.70088
[21]	validation_0-mlogloss:0.69352
[22]	validation_0-mlogloss:0.68341
[23]	validation_0-mlogloss:0.67402
[24]	validation_0-mlogloss:0.66635
[25]	validation_0-mlogloss:0.66120
[26]	validation_0-mlogloss:0.65694
[27]	validation_0-mlogloss:0.64872
[28]	validation_0-mlogloss:0.6

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [85]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
acc = accuracy_score(y_test.astype(int), y_pred)
print(f"Validation Accuracy: {acc:.4f}")


Validation Accuracy: 0.6667


## Live Test with Webcam

In [77]:
import cv2
import mediapipe as mp
import pickle
import cv2
import mediapipe as mp
import pickle
import numpy as np   # <--- add this

def extract_landmarks(landmarks):
    return np.array([[l.x, l.y] for l in landmarks],dtype=np.float32)
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
drawing_utils = mp.solutions.drawing_utils
drawing_styles = mp.solutions.drawing_styles
gesture = ['paper', 'rock', 'scissors','None']
threshold = .8
cap = cv2.VideoCapture(0)  # Open webcam

if not cap.isOpened():
    print("Error: Camera not opened")
    exit()

cv2.namedWindow("Camera")
with open("model.pkl", "rb") as f:
    model = pickle.load(f)
ind = 3 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Flip for a mirror view
        # frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Make predictions
        results = holistic.process(rgb_frame)
        if results.right_hand_landmarks is not None:
            features = extract_landmarks(results.right_hand_landmarks.landmark).reshape(1, -1)
            sample = features[:, :]   # already (1, n_features)
            sample[:, 0::2] -= np.min(sample[:, 0::2], axis=1, keepdims=True)
            sample[:, 1::2] -= np.min(sample[:, 1::2], axis=1, keepdims=True)
            # Predict using the pre-trained ML model
            prediction = model.predict(sample,verbose=0)
            print(prediction)
            if (prediction>threshold).any():
                ind = np.argmax(prediction)
                print(prediction)
            else:
                ind = 3
         
        drawing_utils.draw_landmarks(
            frame,
            results.left_hand_landmarks,
            mp.solutions.holistic.HAND_CONNECTIONS,
            connection_drawing_spec=drawing_styles.get_default_hand_connections_style()
        )

        drawing_utils.draw_landmarks(
            frame,
            results.right_hand_landmarks,
            mp.solutions.holistic.HAND_CONNECTIONS,
            connection_drawing_spec=drawing_styles.get_default_hand_connections_style()
        )
        # # Display prediction
        cv2.putText(frame, f'Gesture Detected: {gesture[ind]}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("Camera", frame)

        if cv2.waitKey(1) & 0xFF == 27:  # ESC to quit
            break

cap.release()
cv2.destroyAllWindows()


[[0.7062915  0.01323527 0.28047323]]
[[0.33678123 0.20225199 0.46096674]]
[[0.96780694 0.00404387 0.02814917]]
[[0.96780694 0.00404387 0.02814917]]
[[0.94363    0.00127911 0.05509096]]
[[0.94363    0.00127911 0.05509096]]
[[9.9016553e-01 4.2114488e-04 9.4133597e-03]]
[[9.9016553e-01 4.2114488e-04 9.4133597e-03]]
[[0.98952675 0.00104948 0.00942382]]
[[0.98952675 0.00104948 0.00942382]]
[[0.49382272 0.31551826 0.19065894]]
[[0.8479698  0.00333344 0.14869684]]
[[0.8479698  0.00333344 0.14869684]]
[[0.78625107 0.00311605 0.21063279]]
[[0.41359803 0.03708206 0.5493199 ]]
[[0.6683413  0.01756286 0.31409585]]
[[0.01225592 0.00190667 0.9858374 ]]
[[0.01225592 0.00190667 0.9858374 ]]
[[3.379086e-03 6.964017e-04 9.959245e-01]]
[[3.379086e-03 6.964017e-04 9.959245e-01]]
[[2.4246161e-04 9.5735240e-06 9.9974805e-01]]
[[2.4246161e-04 9.5735240e-06 9.9974805e-01]]
[[2.3445154e-04 8.3641835e-06 9.9975723e-01]]
[[2.3445154e-04 8.3641835e-06 9.9975723e-01]]
[[2.7359759e-03 6.6485332e-04 9.9659926e-01]]


# Directly Adding to Manual Dataset by Clicking

In [11]:
import cv2
import pickle
import numpy as np
import mediapipe as mp

Holistic = mp.solutions.holistic.Holistic

# Globals
counter = 0
current_frame = None
dataset_file = "data/paper/Manual_dataset_RightHand_paper.pkl"

# Load existing dataset or create empty list
try:
    with open(dataset_file, "rb") as f:
        dataset = pickle.load(f)
except FileNotFoundError:
    dataset = []

def extract_landmarks(landmarks):
    # Access the .landmark attribute which is a list
    return np.array([[l.x, l.y] for l in landmarks.landmark], dtype=np.float32).flatten()

# Mouse callback
def mouse_click(event, x, y, flags, param):
    global counter, current_frame, dataset
    results = param  # param is right_hand_landmarks
    if event == cv2.EVENT_LBUTTONDOWN:
        if current_frame is not None:
            counter += 1
           
            print(counter)

            # Append landmarks if available
            if results is not None:
                ldk_temp = extract_landmarks(results)
                dataset.append(ldk_temp)
                # Save updated dataset
                with open(dataset_file, "wb") as f:
                    pickle.dump(dataset, f)
                print(f"Updated dataset with {len(ldk_temp)} landmarks.")

# Initialize webcam and Mediapipe
cap = cv2.VideoCapture(0)
cv2.namedWindow("Camera")

with Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as mp_holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # frame = cv2.flip(frame, 1)
        current_frame = frame.copy()  # store for saving
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = mp_holistic.process(rgb_frame)
        # Pass right_hand_landmarks to mouse callback
        cv2.setMouseCallback("Camera", mouse_click, param=results.right_hand_landmarks)

        # Draw landmarks for visualization
        if results.right_hand_landmarks:
            mp.solutions.drawing_utils.draw_landmarks(
                frame,
                results.right_hand_landmarks,
                mp.solutions.holistic.HAND_CONNECTIONS
            )

        cv2.imshow("Camera", frame)
        if cv2.waitKey(1) & 0xFF == 27:  # ESC to quit
            break

cap.release()
cv2.destroyAllWindows()


1
Updated dataset with 42 landmarks.
2
Updated dataset with 42 landmarks.
3
Updated dataset with 42 landmarks.
4
Updated dataset with 42 landmarks.
5
Updated dataset with 42 landmarks.
6
Updated dataset with 42 landmarks.
7
Updated dataset with 42 landmarks.
8
Updated dataset with 42 landmarks.
9
Updated dataset with 42 landmarks.
10
Updated dataset with 42 landmarks.
11
Updated dataset with 42 landmarks.
12
Updated dataset with 42 landmarks.
13
Updated dataset with 42 landmarks.
14
Updated dataset with 42 landmarks.
15
Updated dataset with 42 landmarks.
16
Updated dataset with 42 landmarks.
17
Updated dataset with 42 landmarks.
18
Updated dataset with 42 landmarks.
19
Updated dataset with 42 landmarks.
20
Updated dataset with 42 landmarks.
21
Updated dataset with 42 landmarks.
22
Updated dataset with 42 landmarks.
23
Updated dataset with 42 landmarks.
24
Updated dataset with 42 landmarks.
25
Updated dataset with 42 landmarks.
26
Updated dataset with 42 landmarks.
27
Updated dataset wi